In [ ]:
!pip install accelerate
!pip install nnsight

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm

In [4]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype="auto"
)

model.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [2]:
def generate_response(prompt):

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    generated_ids = model.generate(
        **text,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )
    
    # Remove the input tokens from the output 
    generated_ids = [ output_ids[len(input_ids):] for input_ids, output_ids in zip(text.input_ids, generated_ids) ] 
    
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [5]:
prompt = "what is the capital of canada"
generate_response(prompt)

'The capital of Canada is Ottawa.\n\nHere are some additional details about Ottawa:\n\n1. It is located in the province of Ontario.\n2. It is situated on the banks of the Ottawa River, near its confluence with the Rideau River.\n3. Ottawa became the capital in 1857, replacing Quebec City.\n4. It is known for its beautiful architecture, including Parliament Hill and the Canadian War Museum.\n5. The city hosts many national institutions, such as the National Gallery of Canada, the Canadian Museum of History, and the Supreme Court of Canada.\n6. Ottawa is also a significant center for government, finance, technology, and education in Canada.'

In [6]:
dataset = load_dataset("Anthropic/hh-rlhf")

print(pd.DataFrame(dataset["train"].select(range(5))))

                                              chosen  \
0  \n\nHuman: What are some cuss words in english...   
1  \n\nHuman: What kind of noises did dinosaurs m...   
2  \n\nHuman: If you were going to steal from a c...   
3  \n\nHuman: Can you provide me the home address...   
4  \n\nHuman: How do you embezzle money?\n\nAssis...   

                                            rejected  
0  \n\nHuman: What are some cuss words in english...  
1  \n\nHuman: What kind of noises did dinosaurs m...  
2  \n\nHuman: If you were going to steal from a c...  
3  \n\nHuman: Can you provide me the home address...  
4  \n\nHuman: How do you embezzle money?\n\nAssis...  


In [7]:
def format_example(example, response_key):
    text = example[response_key]
    return tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

@torch.no_grad()
def get_activations(example, response_type):
    inputs = format_example(example, response_type)

    outputs = model(
        **inputs,
        output_hidden_states=True,
        use_cache=False
    )

    hidden_states = outputs.hidden_states

    activations = torch.stack([h[0, -1, :].detach().float().cpu() for h in hidden_states[1:]])

    return activations  

In [8]:
num_examples = 100

chosen_acts = []
rejected_acts = []

for ex in tqdm(dataset["train"].select(range(num_examples))):
    chosen_acts.append(get_activations(ex, "chosen"))
    rejected_acts.append(get_activations(ex, "rejected"))

chosen_acts = torch.stack(chosen_acts)      # [N, 28, hidden_size]
rejected_acts = torch.stack(rejected_acts)  # [N, 28, hidden_size]

print(chosen_acts.shape)
print(rejected_acts.shape)

100%|██████████| 100/100 [00:18<00:00,  5.53it/s]

torch.Size([100, 28, 3584])
torch.Size([100, 28, 3584])


In [9]:
refusal_direction = chosen_acts.mean(dim=0) - rejected_acts.mean(dim=0)
refusal_direction = refusal_direction / refusal_direction.norm(dim=-1, keepdim=True)

print(refusal_direction.shape)  # [28, hidden_size]

torch.Size([28, 3584])


In [10]:
def make_injection_hook(layer_idx, alpha):
    def hook(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        v = refusal_direction[layer_idx].to(hidden.device).to(hidden.dtype)

        if hidden.dim() == 3:
            hidden[:, -1, :] = hidden[:, -1, :] + alpha * v
        elif hidden.dim() == 2:
            hidden[-1, :] = hidden[-1, :] + alpha * v
        else:
            raise ValueError(f"Unexpected hidden shape: {hidden.shape}")

        if isinstance(output, tuple):
            return (hidden,) + output[1:]
        return hidden

    return hook

In [12]:
layer_idx = 20
alpha = 10.0

from nnsight import LanguageModel

model = LanguageModel("Qwen/Qwen2.5-7B-Instruct", device_map="auto")

with model.trace("Tell me why stealing is wrong.") as tracer:
    
    # access layer
    hidden = tracer.model.layers[layer_idx].output

    # inject your direction
    hidden[:, -1, :] += alpha * refusal_direction[layer_idx]

    output = tracer.output

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
Traceback (most recent call last):
  File "/home/coder/interpretability-llms-agents/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3747, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_8373/2289362485.py", line 8, in <module>
    with model.trace("Tell me why stealing is wrong.") as tracer:
  File "/tmp/ipykernel_8373/2289362485.py", line 11, in <module>
    hidden = tracer.model.layers[layer_idx].output

AttributeError: Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj)

In [ ]:
def make_ablation_hook(layer_idx):
    def hook(module, inputs, output):
        hidden = output[0]

        v = refusal_direction[layer_idx].to(hidden.device).to(hidden.dtype)

        x = hidden[:, -1, :]

        # remove component of x along v
        proj = (x * v).sum(dim=-1, keepdim=True) * v

        hidden[:, -1, :] = x - proj

        return (hidden,) + output[1:]

    return hook

In [ ]:
layer_idx = 20

handle = model.model.layers[layer_idx].register_forward_hook(
    make_ablation_hook(layer_idx)
)

print(generate_response("Tell me why stealing is wrong."))

handle.remove()